In [ ]:
import matplotlib.pyplot as pyplot
import pandas as pd
import geopandas as gpd
import numpy as np
from google.colab import drive

In [ ]:
drive.mount('/content/drive/')
! pip install geemap # google earth engine
! ls /content/drive/MyDrive/Nino/raw_data

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).
coastline_boundaries	nina34.csv		storm_events
county_boundaries	poverty_rates.xlsm
county_populations.csv	precipitation_and_soil


In [ ]:
import ee
import geemap

ee.Authenticate()
ee.Initialize(project="double-lore-439815-s7")

In [ ]:
merit_hydro = ee.Image('MERIT/Hydro/v1_0_1') # loads the merit hydro dataset
hand = merit_hydro.select('hnd') # specifically selects the HAND feature

risk_threshold_m = 5 # if a given pixel is 5m or less above the nearest stream, it's marked 1; otherwise 0
vulnerable_mask = hand.lte(risk_threshold_m) # turns it into a binary dataset; lte means less than or equal, so everything lte 5 is a 1 and above is 0

# load us county boundaries without alaska, hawaii, or territories
counties = ee.FeatureCollection('TIGER/2018/Counties') \
    .filter(ee.Filter.lt("STATEFP", "60")) \
    .filter(ee.Filter.neq("STATEFP", "02")) \
    .filter(ee.Filter.neq("STATEFP", "15"))

# reduceRegions calculates the percent of each county lying at or beneath our vulnverable zone
# takes the mean of all the values, which are between 0 and 1, giving an exact fraction of vulnerable zones
county_flood_stats = vulnerable_mask.reduceRegions(
    collection=counties,
    reducer=ee.Reducer.mean(),
    scale=90,
    crs='EPSG:5070'
)

hand_df = geemap.ee_to_df(county_flood_stats)
hand_df["hand_vulnerability"] = hand_df["mean"] # rename for readability
hand_df = hand_df[['GEOID', 'hand_vulnerability']]
hand_df

,GEOID,hand_vulnerability
0,29510,0.397792
1,32510,0.224161
2,51580,0.209800
3,51530,0.195965
4,51678,0.202080
...,...,...
3121,44003,0.295109
3122,09007,0.228851
3123,09013,0.181630
3124,09003,0.297104


In [ ]:
hand_df[hand_df["GEOID"] == "13117"]
hand_df.sort_values(by="hand_vulnerability")

,GEOID,hand_vulnerability
284,08019,0.067020
518,16079,0.067579
235,08111,0.070928
1550,30061,0.071117
510,16049,0.072880
...,...,...
334,12051,0.998305
1074,22007,0.998425
1116,22023,0.998653
1092,22057,0.999202


In [ ]:
# generating each county's proximity to the coast
counties = gpd.read_file("/content/drive/MyDrive/Nino/raw_data/county_boundaries/tl_2018_us_county.shp").to_crs(epsg=5070) # equal-area projection
coastlines = gpd.read_file("/content/drive/MyDrive/Nino/raw_data/coastline_boundaries/ne_110m_coastline.shp").to_crs(epsg=5070)

# calculates the minimum distance from every county to the nearest coastline
counties["dist_coast_m"] = counties.geometry.distance(coastlines.unary_union) # unary_union combines all the disjointed coastlines into one shape so that distance can be calculated
counties.sort_values(by="dist_coast_m")

/tmp/ipykernel_56846/4155465986.py:6: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  counties["dist_coast_m"] = counties.geometry.distance(coastlines.unary_union) # unary_union combines all the disjointed coastlines into one shape so that distance can be calculated


,STATEFP,COUNTYFP,COUNTYNS,GEOID,NAME,NAMELSAD,LSAD,CLASSFP,MTFCC,CSAFP,CBSAFP,METDIVFP,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry,dist_coast_m
348,51,119,01480147,51119,Middlesex,Middlesex County,06,H1,G4020,None,None,None,A,337554148,208310589,+37.6069753,-076.5280819,"POLYGON ((1683393.317 1788962.084, 1683382.022...",0.000000e+00
1125,41,057,01135864,41057,Tillamook,Tillamook County,06,H1,G4020,None,None,None,A,2855122951,595798837,+45.4558895,-123.7592988,"POLYGON ((-2153395.205 2837295.009, -2153326.4...",0.000000e+00
2305,72,017,01804488,72017,Barceloneta,Barceloneta Municipio,13,H1,G4020,490,41980,None,A,48421898,31649450,+18.4699891,-066.5582299,"POLYGON ((3172265.413 9128.058, 3172199.526 96...",0.000000e+00
2309,51,193,01480179,51193,Westmoreland,Westmoreland County,06,H1,G4020,None,None,None,A,593950813,61109991,+38.1093110,-076.8039326,"POLYGON ((1648442.281 1841977.044, 1648231.196...",0.000000e+00
256,72,053,01804506,72053,Fajardo,Fajardo Municipio,13,H1,G4020,490,41980,None,A,77448651,195352766,+18.3863776,-065.5884540,"POLYGON ((3271800.266 22879.675, 3271770.246 2...",0.000000e+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1926,69,085,01805245,69085,Northern Islands,Northern Islands Municipality,12,H1,G4020,None,None,None,B,160067129,2939499290,+18.8129658,+145.6569855,"MULTIPOLYGON (((-9911830.427 6585013.865, -991...",1.653320e+06
3209,60,030,01805243,60030,Rose Island,Rose Island,00,H4,G4020,None,None,None,N,82093,147383994,-14.5365298,-168.1512918,"POLYGON ((-9361444.535 79474.033, -9360712.623...",1.664943e+06
891,69,100,01805246,69100,Rota,Rota Municipality,12,H1,G4020,None,None,None,A,85096800,539092797,+14.1524934,+145.2129037,"POLYGON ((-10325973.29 6515922.831, -10324548....",1.724463e+06
967,69,120,01805248,69120,Tinian,Tinian Municipality,12,H1,G4020,None,None,None,A,108237607,632893009,+14.9367835,+145.6010210,"POLYGON ((-10238951.999 6495705, -10237734.885...",1.811648e+06


In [ ]:
counties = counties[["GEOID", "dist_coast_m"]]
counties.sort_values(by="dist_coast_m")

,GEOID,dist_coast_m
348,51119,0.000000e+00
1125,41057,0.000000e+00
2305,72017,0.000000e+00
2309,51193,0.000000e+00
256,72053,0.000000e+00
...,...,...
1926,69085,1.653320e+06
3209,60030,1.664943e+06
891,69100,1.724463e+06
967,69120,1.811648e+06


In [ ]:
# merges the hand and coastline distance datasets using a left merge on the hand df
# this means that all rows in counties that matches hand_df's GEOID's gets added, removing the territories and alaska/hawaii from counties
# since those arent in hand_df
hand_and_coastline_df = hand_df.merge(counties, on="GEOID", how="left")
hand_and_coastline_df.sort_values(by="dist_coast_m")

,GEOID,hand_vulnerability,dist_coast_m
3113,25025,0.475458,0.000000e+00
3125,09011,0.226730,0.000000e+00
2775,06075,0.290434,0.000000e+00
2778,12031,0.888388,0.000000e+00
3120,44005,0.414246,0.000000e+00
...,...,...,...
2110,46071,0.306995,1.555856e+06
2131,46095,0.275423,1.562061e+06
2119,46102,0.294197,1.565391e+06
2117,46121,0.353305,1.587260e+06


In [ ]:
# make sure both datasets are in the same format; they are
print(hand_and_coastline_df["GEOID"].str.len().value_counts())
print(hand_df["GEOID"].dtype, counties["GEOID"].dtype)
hand_and_coastline_df["dist_coast_m"].isna().sum()

GEOID
5    3126
Name: count, dtype: int64
object object


np.int64(0)

In [ ]:
# check the great lakes to ensure thats not being counted as coastline
# correctly labels the great lakes as being far from the shore
hand_and_coastline_df[hand_and_coastline_df["GEOID"] == "26163"]

,GEOID,hand_vulnerability,dist_coast_m
1207,26163,0.900951,650520.472827


In [ ]:
# export data so pipeline doesnt need to be run again
hand_and_coastline_df.to_parquet("/content/drive/MyDrive/Nino/cleaned_data/hand_coast_static.parquet")